# Construção do índice ocupacional (CBO) a partir de `score_results.csv`

Este notebook executa o script `construir_indice_ocupacional_cbo.py` para construir o índice ocupacional e exibir a tabela final.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

def find_project_root() -> Path:
    """Localiza o root do projeto buscando o script em `codigos/`."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for c in candidates:
        if (c / "codigos" / "construir_indice_ocupacional_cbo.py").exists():
            return c
    raise FileNotFoundError(
        "Não foi possível localizar `codigos/construir_indice_ocupacional_cbo.py`."
    )


PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT / "codigos"))

from construir_indice_ocupacional_cbo import build_index

input_csv = PROJECT_ROOT / "dados" / "score_results.csv"
output_csv = PROJECT_ROOT / "resultados" / "tabelas" / "indice_ocupacao_cbo.csv"

try:
    df_indice = build_index(input_path=input_csv, output_csv_path=output_csv)
    print(f"Arquivo gerado: {output_csv}")
    display(df_indice)
except ValueError as e:
    print("Falha na validação dos dados. Detalhes:")
    print(str(e))


## Explicação curta do índice (para a metodologia / resultados)

**1) Como o índice foi construído**
- Para cada combinação de `ocupacao + atividade`, calculamos a **mediana** dos 4 scores disponíveis (2 modelos × 2 rodadas). Essa mediana vira o *score final da atividade*.
- Para cada ocupação, calculamos a **média simples** das 5 medianas das suas atividades. Esse valor é o *escore ocupacional bruto*.
- A normalização para o intervalo **0 a 1** foi feita por divisão: `indice_0_1 = media_das_medianas / 4` (pois o score individual varia de 0 a 4).

**2) Como o desvio padrão foi calculado**
- O `desvio_padrao` de cada ocupação é o desvio padrão das **5 medianas** das atividades que compõem a ocupação. Ele mede a dispersão interna do nível de exposição estimado entre as atividades.

**3) Por que usar mediana no nível da atividade**
- A mediana reduz a influência de valores extremos e representa de forma robusta o *score típico* da atividade ao combinar avaliações de dois modelos e duas rodadas.

**4) Por que normalizar dividindo por 4**
- Como o score original está na escala 0–4, dividir por 4 torna o índice ocupacional comparável em uma escala padronizada de 0 a 1, mantendo a interpretação de intensidade relativa de exposição.